## Example 1: Basic Rioxarray Integration and CRS Management

In [ ]:
# Example 1: Basic Rioxarray Integration and CRS Management
import rioxarray as rxr
import xarray as xr
import numpy as np
from rasterio.crs import CRS

print("=== Rioxarray Integration with Xarray ===")

# Create sample geospatial data array
height, width = 100, 150
data = np.random.randint(0, 255, (height, width), dtype=np.uint8)

# Define spatial coordinates (geographic extent)
lon_min, lon_max = -120.0, -119.0
lat_min, lat_max = 34.0, 35.0

lons = np.linspace(lon_min, lon_max, width)
lats = np.linspace(lat_max, lat_min, height)  # Note: reversed for proper orientation

# Create xarray DataArray
da = xr.DataArray(
    data,
    coords={'y': lats, 'x': lons},
    dims=['y', 'x'],
    name='elevation',
    attrs={
        'description': 'Sample elevation data',
        'units': 'meters'
    }
)

print(f"Created DataArray: {da.shape}")
print(f"Coordinate ranges: lon[{lon_min}, {lon_max}], lat[{lat_min}, {lat_max}]")

# Test rioxarray rio accessor
print(f"\n=== Rioxarray Rio Accessor ===")
print(f"Rio accessor available: {hasattr(da, 'rio')}")

if hasattr(da, 'rio'):
    # Initial CRS status
    print(f"Initial CRS: {da.rio.crs}")
    print(f"Has spatial dimensions: {da.rio.x_dim}, {da.rio.y_dim}")
    
    # Set coordinate reference system
    da_geo = da.rio.set_crs("EPSG:4326")  # WGS84 Geographic
    print(f"CRS assigned: {da_geo.rio.crs}")
    print(f"CRS name: {da_geo.rio.crs.to_string()}")
    
    # Get spatial information
    bounds = da_geo.rio.bounds()
    print(f"Bounds: {bounds}")
    print(f"Width: {da_geo.rio.width}, Height: {da_geo.rio.height}")

# Test different CRS formats
print(f"\n=== CRS Format Support ===")

crs_formats = [
    ("EPSG:4326", "WGS84 Geographic"),
    ("EPSG:3857", "Web Mercator"),
    ("EPSG:32611", "UTM Zone 11N"),
    ("+proj=longlat +datum=WGS84", "Proj4 string")
]

for crs_code, description in crs_formats:
    try:
        test_da = da.rio.set_crs(crs_code)
        print(f"✓ {crs_code}: {description} - {test_da.rio.crs.to_epsg()}")
    except Exception as e:
        print(f"✗ {crs_code}: {description} - Error: {type(e).__name__}")
        raise

# Spatial metadata preservation
print(f"\n=== Spatial Metadata ===")
da_with_crs = da.rio.set_crs("EPSG:4326")

# Transform information
transform = da_with_crs.rio.transform()
if transform:
    print(f"Spatial transform: {transform}")
    print(f"Pixel size: {transform[0]:.6f}, {abs(transform[4]):.6f}")

# Resolution information
resolution = da_with_crs.rio.resolution()
print(f"Resolution (x, y): {resolution}")

print("Basic rioxarray integration and CRS management completed!")

## Example 2: Spatial Clipping and Subsetting with Rioxarray

In [ ]:
# Example 2: Spatial Clipping and Subsetting with Rioxarray
import rioxarray as rxr
import xarray as xr
import numpy as np

print("=== Spatial Clipping and Subsetting ===")

# Create larger geospatial dataset
height, width = 200, 300
data = np.random.randint(0, 1000, (height, width), dtype=np.int16)

# Define California region coordinates
lon_min, lon_max = -125.0, -114.0
lat_min, lat_max = 32.0, 42.0

lons = np.linspace(lon_min, lon_max, width)
lats = np.linspace(lat_max, lat_min, height)

# Create multi-band DataArray (simulate RGB bands)
bands = ['red', 'green', 'blue']
multi_data = np.random.randint(0, 255, (len(bands), height, width), dtype=np.uint8)

da_multiband = xr.DataArray(
    multi_data,
    coords={'band': bands, 'y': lats, 'x': lons},
    dims=['band', 'y', 'x'],
    name='landsat_simulation'
)

# Set CRS
da_multiband = da_multiband.rio.set_crs("EPSG:4326")

print(f"Created multi-band DataArray: {da_multiband.shape}")
print(f"Bands: {list(da_multiband.band.values)}")
print(f"Geographic extent: {da_multiband.rio.bounds()}")

print(f"\n=== Bounding Box Clipping ===")

# Define clipping bounds (Los Angeles area)
la_bounds = (-119.5, 33.5, -117.5, 34.5)  # minx, miny, maxx, maxy
print(f"Clipping to Los Angeles area: {la_bounds}")

# Clip using bounding box
clipped_da = da_multiband.rio.clip_box(*la_bounds)

print(f"Original shape: {da_multiband.shape}")
print(f"Clipped shape: {clipped_da.shape}")
print(f"Clipped bounds: {clipped_da.rio.bounds()}")

# Reduction in data size
original_size = da_multiband.size
clipped_size = clipped_da.size
reduction = (1 - clipped_size / original_size) * 100

print(f"Data reduction: {reduction:.1f}% ({original_size} -> {clipped_size} pixels)")

print(f"\n=== Polygon Clipping ===")

# Create custom polygon geometry
polygon_coords = [
    (-119.0, 34.0),
    (-118.0, 34.0),
    (-118.0, 34.5),
    (-119.0, 34.5),
    (-119.0, 34.0)
]

# rio.clip takes GeoJSON-like geometry mappings (shapely objects work too, but are not required)
polygon_geometry = {'type': 'Polygon', 'coordinates': [polygon_coords]}
polygon_clipped = da_multiband.rio.clip([polygon_geometry], 'EPSG:4326')
polygon_bounds = polygon_clipped.rio.bounds()
print(f"Polygon-clipped shape: {polygon_clipped.shape}")
print(f"Polygon-clipped bounds: {tuple(round(b, 4) for b in polygon_bounds)}")
# the result covers only the pixels whose centres fall inside the polygon
assert polygon_clipped.shape == (3, 10, 27), polygon_clipped.shape
assert polygon_bounds[0] >= -119.01 and polygon_bounds[2] <= -117.98, polygon_bounds
assert polygon_bounds[1] >= 33.98 and polygon_bounds[3] <= 34.51, polygon_bounds
assert polygon_clipped.rio.crs.to_epsg() == 4326

# Alternative: manual coordinate-based clipping
manual_clip_result = da_multiband.sel(
    x=slice(-119.0, -118.0),
    y=slice(34.5, 34.0)
)

print(f"Manual coordinate clipping result: {manual_clip_result.shape}")

print(f"\n=== Advanced Subsetting ===")

# Select specific bands
red_band = da_multiband.sel(band='red')
print(f"Red band selected: {red_band.shape}")

# Spatial subsetting with step
subset_stepped = da_multiband.isel(
    x=slice(0, None, 5),  # Every 5th pixel in x
    y=slice(0, None, 5)   # Every 5th pixel in y
)

print(f"Stepped subset (1:5 sampling): {subset_stepped.shape}")
print(f"Resolution reduction: 25x fewer pixels")

# Regional subset by coordinate values
central_ca = da_multiband.sel(
    x=slice(-122.0, -119.0),
    y=slice(38.0, 36.0)
)

print(f"Central California subset: {central_ca.shape}")

print(f"\n=== Clipping Performance Analysis ===")

# Analyze clipping efficiency
clipping_scenarios = [
    ('Full dataset', da_multiband, 100.0),
    ('LA bounding box', clipped_da, (clipped_da.size / da_multiband.size) * 100),
    ('Single band', red_band, (red_band.size / da_multiband.size) * 100),
    ('Stepped subset', subset_stepped, (subset_stepped.size / da_multiband.size) * 100)
]

print(f"Clipping efficiency comparison:")
for name, data_array, percentage in clipping_scenarios:
    memory_mb = data_array.nbytes / (1024 * 1024)
    print(f"  {name}: {percentage:.1f}% of original ({memory_mb:.2f} MB)")

# Coordinate precision
print(f"\n=== Coordinate Precision ===")
print(f"Original coordinate precision:")
print(f"  X resolution: {np.diff(lons)[0]:.6f} degrees")
print(f"  Y resolution: {np.diff(lats)[0]:.6f} degrees")
print(f"  Approximate pixel size: {abs(np.diff(lons)[0]) * 111320:.1f} meters at equator")

print("Spatial clipping and subsetting completed!")

## Example 3: Coordinate Reprojection and Transformation

In [ ]:
# Example 3: Coordinate Reprojection and Transformation
import rioxarray as rxr
import xarray as xr
import numpy as np
from pyproj import Transformer

print("=== Coordinate Reprojection and Transformation ===")

# Create geographic data in WGS84
height, width = 100, 100
data = np.random.randint(0, 100, (height, width), dtype=np.uint8)

# Define extent around San Francisco Bay Area
lon_min, lon_max = -123.0, -121.0
lat_min, lat_max = 37.0, 38.5

lons = np.linspace(lon_min, lon_max, width)
lats = np.linspace(lat_max, lat_min, height)

# Create DataArray
da_geographic = xr.DataArray(
    data,
    coords={'y': lats, 'x': lons},
    dims=['y', 'x'],
    name='temperature',
    attrs={'units': 'celsius', 'description': 'Surface temperature'}
)

# Set WGS84 geographic CRS
da_geographic = da_geographic.rio.set_crs("EPSG:4326")

print(f"Original geographic data:")
print(f"  Shape: {da_geographic.shape}")
print(f"  CRS: {da_geographic.rio.crs}")
print(f"  Bounds: {da_geographic.rio.bounds()}")

print(f"\n=== Reprojection Analysis ===")

# Define target coordinate reference systems
target_crs_list = [
    ("EPSG:3857", "Web Mercator (Google/OSM)"),
    ("EPSG:32610", "UTM Zone 10N (California)"),
    ("EPSG:3310", "California Albers"),
]

reprojection_results = []

for crs_code, description in target_crs_list:
    print(f"Reprojecting to {crs_code} ({description})...")
    reprojected = da_geographic.rio.reproject(crs_code)
    # the result carries the target CRS in metres, and its extent contains the projected centre of the source area
    assert reprojected.rio.crs.to_epsg() == int(crs_code.split(':')[1]), reprojected.rio.crs
    assert reprojected.rio.crs.linear_units == 'metre', reprojected.rio.crs.linear_units
    reprojected_bounds = reprojected.rio.bounds()
    centre_x, centre_y = Transformer.from_crs('EPSG:4326', crs_code, always_xy=True).transform(-122.0, 37.75)
    assert reprojected_bounds[0] < centre_x < reprojected_bounds[2], (reprojected_bounds, centre_x)
    assert reprojected_bounds[1] < centre_y < reprojected_bounds[3], (reprojected_bounds, centre_y)
    result_info = {
        'crs': crs_code,
        'description': description,
        'success': True,
        'original_bounds': da_geographic.rio.bounds(),
        'reprojected_bounds': reprojected_bounds,
        'units': 'meters' if crs_code != 'EPSG:4326' else 'degrees'
    }
    
    # Simulate coordinate transformation effects
    if crs_code == "EPSG:3857":  # Web Mercator
        # Web Mercator distorts area significantly at high latitudes
        result_info.update({
            'distortion_type': 'area',
            'suitable_for': 'web mapping, navigation',
            'distortion_level': 'moderate at mid-latitudes'
        })
    elif crs_code == "EPSG:32610":  # UTM
        # UTM preserves area and distance well within zone
        result_info.update({
            'distortion_type': 'minimal within zone',
            'suitable_for': 'engineering, surveying',
            'distortion_level': 'low'
        })
    elif crs_code == "EPSG:3310":  # California Albers
        # Albers preserves area
        result_info.update({
            'distortion_type': 'area-preserving',
            'suitable_for': 'regional analysis',
            'distortion_level': 'low for California'
        })
    
    reprojection_results.append(result_info)
    print(f"  ✓ Success: {description}")
    print(f"  ✓ Units: {result_info['units']}")
    print(f"  ✓ Suitable for: {result_info.get('suitable_for', 'general use')}")

assert [r['crs'] for r in reprojection_results] == [c for c, _ in target_crs_list], reprojection_results
print(f"\n=== Coordinate System Properties ===")

# Analyze coordinate system properties
for result in reprojection_results:
    print(f"{result['crs']} - {result['description']}:")
    print(f"  Distortion: {result.get('distortion_level', 'varies')}")
    print(f"  Best for: {result.get('suitable_for', 'general mapping')}")
    print(f"  Units: {result['units']}")
    print()

print(f"=== Reprojection Workflow Simulation ===")

# Simulate typical reprojection workflow
workflow_steps = [
    "1. Load geospatial data with geographic coordinates",
    "2. Verify and set source CRS (EPSG:4326)",
    "3. Choose appropriate target CRS based on analysis needs",
    "4. Reproject data using rio.reproject()",
    "5. Verify reprojected coordinates and bounds",
    "6. Update metadata and attributes"
]

print("Typical reprojection workflow:")
for step in workflow_steps:
    print(f"  {step}")

print(f"\n=== Resolution and Resampling ===")

# Simulate resolution changes during reprojection
geographic_res = abs(np.diff(lons)[0])  # degrees
print(f"Original resolution: {geographic_res:.6f} degrees")
print(f"Approximate ground resolution: {geographic_res * 111320:.0f} meters at equator")

# Different resampling methods for reprojection
resampling_methods = [
    ("nearest", "Preserves original values, good for categorical data"),
    ("bilinear", "Smooth interpolation, good for continuous data"),
    ("cubic", "High-quality interpolation, slower processing"),
    ("average", "Area-weighted average, good for downsampling"),
    ("mode", "Most frequent value, good for categorical data")
]

print(f"\nResampling methods for reprojection:")
for method, description in resampling_methods:
    print(f"  {method}: {description}")

print(f"\n=== Spatial Reference Validation ===")

# Validate spatial reference consistency
validation_checks = [
    ("CRS definition", "EPSG:4326 correctly defined"),
    ("Coordinate bounds", f"Within expected geographic range"),
    ("Coordinate order", "Longitude (x), Latitude (y)"),
    ("Units consistency", "Degrees for geographic, meters for projected"),
    ("Datum compatibility", "WGS84 datum alignment")
]

print("Spatial reference validation checklist:")
for check, status in validation_checks:
    print(f"  ✓ {check}: {status}")

# Coordinate transformation matrix simulation
print(f"\n=== Transformation Mathematics ===")
print("Geographic to projected coordinate transformation involves:")
print("  - Datum transformation (if needed)")
print("  - Map projection equations")
print("  - Coordinate system rotation")
print("  - Scale factor application")
print("  - Translation to false origin")

print("Coordinate reprojection and transformation completed!")

## Example 4: Raster I/O and File Format Handling

In [ ]:
# Example 4: Raster I/O and File Format Handling  
import rioxarray as rxr
import xarray as xr
import numpy as np
import pandas as pd
import tempfile
import os

print("=== Raster I/O and File Format Handling ===")

# Create sample multi-dimensional geospatial dataset
times = ['2023-01-01', '2023-02-01', '2023-03-01']
bands = ['B1', 'B2', 'B3', 'B4']  # Blue, Green, Red, NIR
height, width = 50, 75

# Create 4D data array (time, band, y, x)
data = np.random.randint(0, 4096, (len(times), len(bands), height, width), dtype=np.uint16)

# Define spatial coordinates  
lon_min, lon_max = -118.5, -117.5
lat_min, lat_max = 34.0, 34.5

lons = np.linspace(lon_min, lon_max, width)
lats = np.linspace(lat_max, lat_min, height)

# Create xarray Dataset
ds = xr.Dataset({
    'reflectance': xr.DataArray(
        data,
        coords={
            'time': pd.to_datetime(times),
            'band': bands,
            'y': lats,
            'x': lons
        },
        dims=['time', 'band', 'y', 'x'],
        attrs={
            'description': 'Surface reflectance',
            'units': 'digital numbers',
            'scale_factor': 0.0001,
            'add_offset': 0
        }
    )
})

# Set CRS for the dataset
ds = ds.rio.set_crs("EPSG:4326")

print(f"Created multi-dimensional dataset:")
print(f"  Dimensions: {dict(ds.dims)}")
print(f"  Data variables: {list(ds.data_vars)}")
print(f"  CRS: {ds.rio.crs}")

print(f"\n=== File Format Support ===")

# Supported raster formats with rioxarray
raster_formats = [
    {
        'format': 'GeoTIFF',
        'extension': '.tif',
        'description': 'Tagged Image File Format with geospatial metadata',
        'advantages': ['Widely supported', 'Embedded CRS', 'Compression options'],
        'use_cases': ['Satellite imagery', 'DEMs', 'Classification maps']
    },
    {
        'format': 'NetCDF',
        'extension': '.nc',
        'description': 'Network Common Data Form for scientific data',
        'advantages': ['Multi-dimensional', 'Self-describing', 'CF conventions'],
        'use_cases': ['Climate data', 'Ocean models', 'Time series analysis']
    },
    {
        'format': 'COG',
        'extension': '.tif',
        'description': 'Cloud Optimized GeoTIFF for web streaming',
        'advantages': ['HTTP range requests', 'Overviews', 'Tiled structure'],
        'use_cases': ['Web mapping', 'Cloud analytics', 'Large datasets']
    },
    {
        'format': 'Zarr',
        'extension': '.zarr',
        'description': 'Chunked, compressed, N-dimensional arrays',
        'advantages': ['Cloud native', 'Parallel I/O', 'Compression'],
        'use_cases': ['Big data', 'Distributed computing', 'Time series']
    }
]

print("Supported raster formats:")
for fmt in raster_formats:
    print(f"  {fmt['format']} ({fmt['extension']}):")
    print(f"    Description: {fmt['description']}")
    print(f"    Advantages: {', '.join(fmt['advantages'])}")
    print(f"    Use cases: {', '.join(fmt['use_cases'])}")
    print()

print(f"=== Reading Raster Data ===")

# Simulate reading different data types
read_examples = [
    {
        'file_type': 'Single-band GeoTIFF',
        'method': 'rxr.open_rasterio("elevation.tif")',
        'result_dims': ['band', 'y', 'x'],
        'typical_size': '1 x 1000 x 1000'
    },
    {
        'file_type': 'Multi-band Landsat',
        'method': 'rxr.open_rasterio("landsat.tif")',
        'result_dims': ['band', 'y', 'x'],
        'typical_size': '7 x 7000 x 7000'
    },
    {
        'file_type': 'NetCDF time series',
        'method': 'xr.open_dataset("climate.nc")',
        'result_dims': ['time', 'lat', 'lon'],
        'typical_size': '365 x 180 x 360'
    },
    {
        'file_type': 'COG with overviews',
        'method': 'rxr.open_rasterio("image.cog.tif")',
        'result_dims': ['band', 'y', 'x'],
        'typical_size': '3 x 10000 x 10000 + overviews'
    }
]

print("Reading raster data examples:")
for example in read_examples:
    print(f"  {example['file_type']}:")
    print(f"    Method: {example['method']}")
    print(f"    Dimensions: {example['result_dims']}")
    print(f"    Typical size: {example['typical_size']}")

print(f"\n=== Advanced I/O Operations ===")

# Advanced I/O patterns
io_patterns = {
    'lazy_loading': {
        'description': 'Load metadata only, data loaded on demand',
        'benefit': 'Memory efficient for large datasets',
        'example': 'da = rxr.open_rasterio("large.tif", chunks={"x": 1000, "y": 1000})'
    },
    'chunked_processing': {
        'description': 'Process data in chunks for memory management',
        'benefit': 'Handle datasets larger than RAM',
        'example': 'result = da.chunk({"x": 500, "y": 500}).mean(dim="band")'
    },
    'band_selection': {
        'description': 'Read only specific bands from multi-band files',
        'benefit': 'Reduce I/O and memory usage',
        'example': 'da = rxr.open_rasterio("image.tif", band_as_variable=True)["B4"]'
    },
    'spatial_windowing': {
        'description': 'Read only spatial subsets of large files',
        'benefit': 'Fast access to regions of interest',
        'example': 'da = rxr.open_rasterio("image.tif").rio.clip_box(minx, miny, maxx, maxy)'
    }
}

for pattern, details in io_patterns.items():
    print(f"{pattern.replace('_', ' ').title()}:")
    print(f"  Description: {details['description']}")
    print(f"  Benefit: {details['benefit']}")
    print(f"  Example: {details['example']}")
    print()

print(f"=== Performance Considerations ===")

# I/O performance factors
performance_factors = [
    ('File format', 'GeoTIFF good for single images, NetCDF for time series'),
    ('Compression', 'LZW for lossless, JPEG for lossy, depends on data type'),
    ('Chunking', 'Match chunk size to access patterns and memory constraints'),
    ('Tiling', 'Tiled formats (COG) enable efficient spatial access'),
    ('Overviews', 'Pyramid overviews speed up display at multiple scales'),
    ('Data type', 'Use appropriate precision (uint8 vs uint16 vs float32)'),
    ('Coordinate precision', 'Balance accuracy with file size')
]

print("I/O performance optimization factors:")
for factor, recommendation in performance_factors:
    print(f"  {factor}: {recommendation}")

print("Raster I/O and file format handling completed!")

## Example 5: Advanced Geospatial Analysis and Production Workflows

In [ ]:
# Example 5: Advanced Geospatial Analysis and Production Workflows
import rioxarray as rxr
import xarray as xr
import numpy as np
import pandas as pd

print("=== Advanced Geospatial Analysis with Rioxarray ===")

# Create multi-temporal satellite imagery simulation
times = pd.date_range('2023-01-01', '2023-12-01', freq='MS')
bands = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
height, width = 100, 150

# Simulate Landsat-like reflectance data
data = np.random.randint(500, 3000, (len(times), len(bands), height, width), dtype=np.uint16)

# Add seasonal variation
for t_idx, time in enumerate(times):
    season_factor = 1 + 0.3 * np.sin(2 * np.pi * t_idx / 12)  # Annual cycle
    data[t_idx] = (data[t_idx] * season_factor).astype(np.uint16)

# Define coordinates (Central Valley, California)
lon_min, lon_max = -121.5, -120.0
lat_min, lat_max = 36.5, 37.5

lons = np.linspace(lon_min, lon_max, width)
lats = np.linspace(lat_max, lat_min, height)

# Create time series Dataset
satellite_ds = xr.Dataset({
    'reflectance': xr.DataArray(
        data,
        coords={
            'time': times,
            'band': bands,
            'y': lats,
            'x': lons
        },
        dims=['time', 'band', 'y', 'x'],
        attrs={
            'description': 'Surface reflectance time series',
            'units': 'scaled reflectance',
            'scale_factor': 0.0001,
            'sensor': 'simulated_landsat'
        }
    )
})

# Set CRS and spatial metadata
satellite_ds = satellite_ds.rio.set_crs("EPSG:4326")
satellite_ds = satellite_ds.rio.set_spatial_dims(x_dim='x', y_dim='y')

print(f"Multi-temporal dataset created:")
print(f"  Time range: {times[0].strftime('%Y-%m-%d')} to {times[-1].strftime('%Y-%m-%d')}")
print(f"  Temporal resolution: {len(times)} monthly observations")
print(f"  Spectral bands: {len(bands)} bands")
print(f"  Spatial dimensions: {height} x {width} pixels")
print(f"  Geographic extent: {satellite_ds.rio.bounds()}")

print(f"\n=== Vegetation Index Calculation ===")

# Calculate common vegetation indices
def calculate_ndvi(ds):
    """Calculate Normalized Difference Vegetation Index"""
    red = ds.reflectance.sel(band='red') * 0.0001
    nir = ds.reflectance.sel(band='nir') * 0.0001
    ndvi = (nir - red) / (nir + red)
    return ndvi.where(ndvi.notnull(), -9999)

def calculate_evi(ds):
    """Calculate Enhanced Vegetation Index"""
    blue = ds.reflectance.sel(band='blue') * 0.0001
    red = ds.reflectance.sel(band='red') * 0.0001
    nir = ds.reflectance.sel(band='nir') * 0.0001
    
    evi = 2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1)
    return evi.where(evi.notnull(), -9999)

def calculate_ndwi(ds):
    """Calculate Normalized Difference Water Index"""
    green = ds.reflectance.sel(band='green') * 0.0001
    nir = ds.reflectance.sel(band='nir') * 0.0001
    ndwi = (green - nir) / (green + nir)
    return ndwi.where(ndwi.notnull(), -9999)

# Calculate vegetation indices
ndvi = calculate_ndvi(satellite_ds)
evi = calculate_evi(satellite_ds)
ndwi = calculate_ndwi(satellite_ds)

print(f"Vegetation indices calculated:")
print(f"  NDVI range: {ndvi.min().values:.3f} to {ndvi.max().values:.3f}")
print(f"  EVI range: {evi.min().values:.3f} to {evi.max().values:.3f}")
print(f"  NDWI range: {ndwi.min().values:.3f} to {ndwi.max().values:.3f}")

print(f"\n=== Time Series Analysis ===")

# Temporal statistics
temporal_stats = {
    'ndvi_mean': ndvi.mean(dim='time'),
    'ndvi_std': ndvi.std(dim='time'),
    'ndvi_trend': ndvi.polyfit(dim='time', deg=1).polyfit_coefficients.sel(degree=1),
    'seasonal_amplitude': ndvi.max(dim='time') - ndvi.min(dim='time')
}

for stat_name, stat_data in temporal_stats.items():
    print(f"{stat_name.replace('_', ' ').title()}:")
    print(f"  Data range: {stat_data.min().values:.4f} to {stat_data.max().values:.4f}")
    print(f"  Mean value: {stat_data.mean().values:.4f}")

print(f"\n=== Spatial Analysis ===")

# Spatial aggregation and analysis
spatial_analyses = []

# Regional statistics
regions = [
    {'name': 'Northern', 'y_slice': slice(37.2, 37.5), 'x_slice': slice(-121.5, -120.8)},
    {'name': 'Central', 'y_slice': slice(36.9, 37.2), 'x_slice': slice(-121.5, -120.8)},
    {'name': 'Southern', 'y_slice': slice(36.5, 36.9), 'x_slice': slice(-121.5, -120.8)}
]

for region in regions:
    regional_ndvi = ndvi.sel(y=region['y_slice'], x=region['x_slice'])
    regional_mean = regional_ndvi.mean(dim=['x', 'y'])
    
    spatial_analyses.append({
        'region': region['name'],
        'mean_ndvi': regional_mean.mean().values,
        'seasonal_range': (regional_mean.max() - regional_mean.min()).values,
        'pixel_count': regional_ndvi.isel(time=0).size
    })

print(f"Regional NDVI analysis:")
for analysis in spatial_analyses:
    print(f"  {analysis['region']} Region:")
    print(f"    Mean NDVI: {analysis['mean_ndvi']:.3f}")
    print(f"    Seasonal range: {analysis['seasonal_range']:.3f}")
    print(f"    Pixel count: {analysis['pixel_count']}")

print(f"\n=== Production Workflow Integration ===")

# Production workflow components
workflow_components = {
    'data_ingestion': {
        'description': 'Automated satellite data download and preprocessing',
        'tools': ['rioxarray', 'requests', 'boto3'],
        'frequency': 'Daily/Weekly',
        'output_format': 'Cloud Optimized GeoTIFF'
    },
    'quality_assessment': {
        'description': 'Cloud masking, atmospheric correction validation',
        'tools': ['rioxarray', 'scipy', 'scikit-image'],
        'metrics': ['cloud_cover', 'data_completeness', 'geometric_accuracy'],
        'thresholds': {'cloud_cover': '<20%', 'completeness': '>95%'}
    },
    'index_calculation': {
        'description': 'Vegetation and environmental index computation',
        'indices': ['NDVI', 'EVI', 'NDWI', 'SAVI', 'LAI'],
        'processing': 'Parallel with Dask',
        'validation': 'Statistical outlier detection'
    },
    'temporal_analysis': {
        'description': 'Time series analysis and trend detection',
        'methods': ['seasonal_decomposition', 'change_detection', 'phenology'],
        'outputs': ['trend_maps', 'anomaly_detection', 'seasonal_metrics']
    },
    'delivery': {
        'description': 'Product generation and distribution',
        'formats': ['GeoTIFF', 'NetCDF', 'Zarr', 'COG'],
        'services': ['Web map services', 'APIs', 'Data portals'],
        'users': ['Researchers', 'Farmers', 'Government agencies']
    }
}

print("Production workflow components:")
for component, details in workflow_components.items():
    print(f"  {component.replace('_', ' ').title()}:")
    print(f"    Description: {details['description']}")
    if 'tools' in details:
        print(f"    Tools: {', '.join(details['tools'])}")
    if 'frequency' in details:
        print(f"    Frequency: {details['frequency']}")
    if 'indices' in details:
        print(f"    Indices: {', '.join(details['indices'])}")
    print()

print(f"=== Performance and Scalability ===")

# Performance optimization strategies
optimization_strategies = [
    {
        'strategy': 'Chunked Processing',
        'description': 'Process data in spatial/temporal chunks',
        'benefit': 'Memory efficiency for large datasets',
        'implementation': 'Use xarray/dask chunking with optimal chunk sizes'
    },
    {
        'strategy': 'Lazy Loading',
        'description': 'Load data only when needed',
        'benefit': 'Faster initialization and lower memory usage',
        'implementation': 'Use rioxarray.open_rasterio with chunks parameter'
    },
    {
        'strategy': 'Parallel Computing',
        'description': 'Distribute computation across cores/nodes',
        'benefit': 'Faster processing of large time series',
        'implementation': 'Integrate with Dask for parallel processing'
    },
    {
        'strategy': 'Format Optimization',
        'description': 'Use efficient storage formats',
        'benefit': 'Faster I/O and smaller file sizes',
        'implementation': 'COG for spatial access, Zarr for time series'
    }
]

print("Performance optimization strategies:")
for strategy in optimization_strategies:
    print(f"  {strategy['strategy']}:")
    print(f"    Description: {strategy['description']}")
    print(f"    Benefit: {strategy['benefit']}")
    print(f"    Implementation: {strategy['implementation']}")
    print()

# Memory usage simulation
total_pixels = len(times) * len(bands) * height * width
memory_usage_mb = total_pixels * 2 / (1024 * 1024)  # 2 bytes per uint16
print(f"Dataset memory footprint:")
print(f"  Total pixels: {total_pixels:,}")
print(f"  Memory usage: {memory_usage_mb:.1f} MB")
print(f"  Recommended chunk size: {int(np.sqrt(50 * 1024 * 1024 / (len(bands) * 2)))} x {int(np.sqrt(50 * 1024 * 1024 / (len(bands) * 2)))} pixels")

print("Advanced geospatial analysis and production workflows completed!")